In [ ]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [ ]:
%pip install pyyaml

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Automatically get current user email
username = w.current_user.me().user_name

# Automatically get workspace host URL
workspace_host = w.config.host

# Automatically construct output directory
WORKSPACE_HOST = workspace_host
ROOT_PATH = f'/Users/{username}/.bundle/${{bundle.name}}/${{bundle.target}}'

# Group-Based Configuration Examples

This notebook demonstrates how to use group-based defaults and overrides to apply different configuration values to different pipeline groups from a single CSV file.

## Matching Precedence

Config keys are matched in this order (most specific wins):
1. `pipeline_group` (prefix_subgroup) - e.g., `'sales_2'`
2. `prefix` - e.g., `'sales'`
3. `project_name` - e.g., `'my_project'`
4. `'*'` (global)

In [ ]:
from core.runner import run_pipeline_generation

targets = {
    'dev': {
        'workspace_host': WORKSPACE_HOST,
        'root_path': ROOT_PATH
    }
}

---
# SaaS Connector Examples (Salesforce)

## Example 1: Global Defaults

Apply the same default values to all rows using a flat dictionary format.

This is the simplest form - all pipelines get the same schedule.

In [ ]:
input_csv = '01_global_defaults/pipeline_config.csv'
output_dir = f'/Workspace/Users/{username}/lakeflow-tapworks/examples/group_based_config/01_global_defaults/deployment'

# Simple flat dict - applies to ALL rows
default_values = {
    'schedule': '0 */6 * * *',  # Every 6 hours
}

result_df = run_pipeline_generation(
    connector_name='salesforce',
    input_source=input_csv,
    output_dir=output_dir,
    targets=targets,
    default_values=default_values,
)

print("Results:")
print(result_df[['source_table_name', 'pipeline_group', 'schedule']].to_string(index=False))

---
## Example 2: Prefix-Based Configuration

Apply different defaults based on the `prefix` column.

Use case: Different teams (sales, hr, finance) need different schedules.

In [ ]:
input_csv = '02_prefix_based/pipeline_config.csv'
output_dir = f'/Workspace/Users/{username}/lakeflow-tapworks/examples/group_based_config/02_prefix_based/deployment'

# Group-based defaults - different values per prefix
default_values = {
    '*': {'schedule': '0 */6 * * *'},       # Global default: every 6 hours
    'sales': {'schedule': '*/15 * * * *'},  # Sales: every 15 minutes
    'hr': {'schedule': '0 0 * * *'},        # HR: daily at midnight
    # finance uses global default (every 6 hours)
}

result_df = run_pipeline_generation(
    connector_name='salesforce',
    input_source=input_csv,
    output_dir=output_dir,
    targets=targets,
    default_values=default_values,
)

print("Results:")
print(result_df[['source_table_name', 'prefix', 'pipeline_group', 'schedule']].to_string(index=False))

---
## Example 3: Pipeline Group Specific Configuration

Apply different defaults based on `pipeline_group` (prefix_subgroup) for the most granular control.

Use case: Different subgroups within the same prefix need different schedules.

In [ ]:
input_csv = '03_pipeline_group_specific/pipeline_config.csv'
output_dir = f'/Workspace/Users/{username}/lakeflow-tapworks/examples/group_based_config/03_pipeline_group_specific/deployment'

# Group-based defaults with pipeline_group specificity
default_values = {
    '*': {'schedule': '0 */6 * * *'},        # Global default
    'sales': {'schedule': '*/15 * * * *'},   # All sales_* groups
    'sales_2': {'schedule': '*/30 * * * *'}, # Only sales_2 (overrides 'sales')
    'sales_3': {'schedule': '0 * * * *'},    # Only sales_3 (overrides 'sales')
}

result_df = run_pipeline_generation(
    connector_name='salesforce',
    input_source=input_csv,
    output_dir=output_dir,
    targets=targets,
    default_values=default_values,
)

print("Results:")
print(result_df[['source_table_name', 'prefix', 'subgroup', 'pipeline_group', 'schedule']].to_string(index=False))

---
## Example 4: Group-Based Overrides

Overrides work the same way as defaults, but they **overwrite** values instead of just filling missing ones.

Use case: Pause specific pipeline groups (e.g., for compliance review).

In [ ]:
input_csv = '02_prefix_based/pipeline_config.csv'
output_dir = f'/Workspace/Users/{username}/lakeflow-tapworks/examples/group_based_config/02_prefix_based/deployment_with_overrides'

# Defaults for schedule
default_values = {
    '*': {'schedule': '*/15 * * * *'},
}

# Overrides for pause_status - finance pipelines are paused
override_config = {
    '*': {'pause_status': 'UNPAUSED'},
    'finance': {'pause_status': 'PAUSED'},
}

result_df = run_pipeline_generation(
    connector_name='salesforce',
    input_source=input_csv,
    output_dir=output_dir,
    targets=targets,
    default_values=default_values,
    override_config=override_config,
)

print("Results:")
print(result_df[['source_table_name', 'prefix', 'schedule', 'pause_status']].to_string(index=False))

---
# Database Connector Examples (SQL Server)

## Example 5: Database Pipelines with Group-Based Schedules

Apply different schedules to different pipeline groups for SQL Server ingestion.

Use case: Sales data needs frequent updates, HR data is updated daily.

In [ ]:
input_csv = '04_database_pipelines/pipeline_config.csv'
output_dir = f'/Workspace/Users/{username}/lakeflow-tapworks/examples/group_based_config/04_database_pipelines/deployment'

# Group-based defaults for database pipelines
default_values = {
    '*': {'schedule': '0 */6 * * *'},       # Global: every 6 hours
    'sales': {'schedule': '*/15 * * * *'},  # Sales: every 15 minutes
    'hr': {'schedule': '0 0 * * *'},        # HR: daily at midnight
    # finance uses global default
}

result_df = run_pipeline_generation(
    connector_name='sql_server',
    input_source=input_csv,
    output_dir=output_dir,
    targets=targets,
    default_values=default_values,
)

print("Results:")
print(result_df[['source_table_name', 'prefix', 'pipeline_group', 'gateway', 'schedule']].to_string(index=False))

---
## Example 6: Database Gateways with Group-Based Configuration

Apply different configurations to different gateways.

Use case: Different gateways for different source databases, with different schedules per pipeline group.

In [ ]:
input_csv = '05_database_gateways/pipeline_config.csv'
output_dir = f'/Workspace/Users/{username}/lakeflow-tapworks/examples/group_based_config/05_database_gateways/deployment'

# Group-based defaults with pipeline_group specificity
default_values = {
    '*': {'schedule': '0 */6 * * *'},        # Global: every 6 hours
    'sales': {'schedule': '*/15 * * * *'},   # All sales pipelines
    'sales_2': {'schedule': '*/30 * * * *'}, # sales_2 specifically (less frequent)
    'hr': {'schedule': '0 0 * * *'},         # HR: daily
    'finance': {'schedule': '0 */12 * * *'}, # Finance: every 12 hours
}

# Override to pause finance pipelines
override_config = {
    '*': {'pause_status': 'UNPAUSED'},
    'finance': {'pause_status': 'PAUSED'},
}

result_df = run_pipeline_generation(
    connector_name='sql_server',
    input_source=input_csv,
    output_dir=output_dir,
    targets=targets,
    default_values=default_values,
    override_config=override_config,
)

print("Results:")
print(result_df[['source_table_name', 'prefix', 'subgroup', 'pipeline_group', 'gateway', 'schedule', 'pause_status']].to_string(index=False))

---
# Summary

## Config Key Matching

| Format | Example | Matches |
|--------|---------|--------|
| Flat dict | `{'schedule': '...'}` | All rows |
| Global | `{'*': {'schedule': '...'}}` | All rows (fallback) |
| Prefix | `{'sales': {'schedule': '...'}}` | Rows where `prefix='sales'` |
| Pipeline group | `{'sales_2': {'schedule': '...'}}` | Rows where `pipeline_group='sales_2'` |

## Defaults vs Overrides

| Parameter | Behavior |
|-----------|----------|
| `default_values` | Fill missing/empty values only |
| `override_config` | Overwrite all values |

## Connector Support

Group-based configuration works with all connectors:
- **SaaS**: Salesforce, ServiceNow, Google Analytics, Workday Reports
- **Database**: SQL Server, PostgreSQL